In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
import seaborn as sns

# Set style for better plots
plt.style.use('default')
sns.set_palette("muted")

FIGSIZE = (3,3)
DPI = 200
TITLE_SIZE = 11

# Read the CSV file
df = pd.read_csv("./trial2_train_pre.csv")
df_test = pd.read_csv("./trial2_NACC_single_pre.csv")

# Create custom colormap for labels
visit_colors = {
    1: 'green',
    2: 'steelblue',
    # 3: 'brown',
    4: 'purple',
}
subj_colors = {
    1: 'c',       # Both first and last points have label 1
    2: 'orange',   # Both first and last points have label 2 or 3
    3: 'saddlebrown',      # First point has label 2 or 3, last point has label 4
    4: 'm',         # Both first and last points have label 4
}

labels = {
    1: 'CN',
    2: 'MCI',
    # 3: 'LMCI',
    4: 'AD'
}
subj_labels = {
    1: 'CN → CN',
    2: 'MCI → MCI',
    3: 'MCI → AD',
    4: 'AD → AD'
}

# Preprocess: Get the subject-wise labels
def assign_subject_label(group):
    lb_values = group['lb']

    # If all lb are 4, assign subj_lb as 4
    if (lb_values == 4).all():
        return 4
    # If all lb are 1, assign subj_lb as 1
    elif (lb_values == 1).all():
        return 1
    # If any lb is 4 (not all), assign subj_lb as 3
    elif (lb_values == 4).any():
        return 3
    # Otherwise, assign subj_lb as 2
    elif (lb_values == 2).any() or (lb_values == 3).any():
        return 2
    else:
        return -1

subject_labels = df.groupby('RID').apply(assign_subject_label).reset_index()
subject_labels.columns = ['RID', 'subj_lb']
subject_labels_test = df_test.groupby('RID').apply(assign_subject_label).reset_index()
subject_labels_test.columns = ['RID', 'subj_lb']

# Merge back to the original dataframe
df = df.merge(subject_labels, on='RID', how='left')
df_test = df_test.merge(subject_labels_test, on='RID', how='left')

# --- Points Plot ---

## All Visits

In [ ]:
# Figure 1-1: Plot all visits colored by lb with custom colors (using sorted dataframe)
# Plot each label separately to create proper legend entries
def draw_all_visits(dataset):
    print(f"All Visits ({dataset})")
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test
    
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    label_order = [2, 1, 4]
    for i in label_order:
        mask = df_tmp['lb'] == i
        if mask.any():
            ax.scatter(df_tmp[mask]['embedding1'], df_tmp[mask]['embedding2'], 
                    c=visit_colors[i], alpha=0.7, s=10, linewidths=0, label=f'{labels[i]}')

    ax.set_title('All Visits', fontsize=TITLE_SIZE)

    # Remove scales (tick marks and labels) from both axes
    ax.set_xticks([])
    ax.set_yticks([])

    # Add legend inside the plot
    lgd_handles, lgd_labels = ax.get_legend_handles_labels()
    # order = [3, 1, 2, 4]
    order = [1, 2, 3]
    lgd_handles = [lgd_handles[i-1] for i in order]
    lgd_labels = [lgd_labels[i-1] for i in order]
    ax.legend(lgd_handles, lgd_labels, loc='upper right', fontsize=10)

    fig.tight_layout()
    fig.show()

# draw_all_visits('train')
draw_all_visits('test')

## Last visits by subject

In [ ]:
# Figure 1-3,4,5,6,7,8: Plot last points of each subject. Base on subj_id.

def plot_by_subj_lb(dataset, subj_lb):
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Plot all points as grey background
    ax.scatter(df_tmp['embedding1'], df_tmp['embedding2'],
               c='lightgrey', alpha=0.3, s=8, linewidths=0, label='All data')

    # Get only the last points for each subject
    last_points_df = df_tmp  # .loc[df_tmp.groupby('RID')['age'].idxmax()]

    mask = last_points_df['subj_lb'] == subj_lb
    if mask.any():
        ax.scatter(last_points_df[mask]['embedding1'], last_points_df[mask]['embedding2'],
                   c=visit_colors[subj_lb], alpha=0.7, s=10, linewidths=0, label=f'{labels[subj_lb]}')

    ax.set_title(f'{labels[subj_lb]}', fontsize=TITLE_SIZE)

    # Remove scales (tick marks and labels) from both axes
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(fontsize=10, loc='upper right')

    fig.tight_layout()
    fig.show()

# plot_by_subj_lb('train', 1)
plot_by_subj_lb('test', 1)
# plot_by_subj_lb('train', 2)
plot_by_subj_lb('test', 2)
# plot_by_subj_lb('train', 3)
# plot_by_subj_lb('test', 3)
# plot_by_subj_lb('train', 4)
plot_by_subj_lb('test', 4)

## Clusters

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def visualize_trajectory(dataset):
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test
    
    # Unique clusters and centers
    cluster_centers = df_tmp.groupby(["cluster", "branch"], as_index=False)[["embedding1", "embedding2"]].mean()

    # Branch ordering by cluster
    branch_order = (
        df_tmp.groupby(["cluster", "branch"], as_index=False)["cluster"]
        .mean()
        .sort_values("cluster")
    )

    # Merge ordering into centers
    cluster_centers = cluster_centers.merge(branch_order, on=["cluster", "branch"])

    # Plot setup
    fig, ax = plt.subplots(figsize=(3, 2), dpi=DPI)
    
    # Get all clusters in order
    clusters = df_tmp["cluster"].unique()
    clusters.sort()
    n_clusters = len(clusters)

    # Create branch-specific color mapping
    def get_branch_color(branch, cluster_idx, total_clusters_in_branch):
        if branch == "normal_branch":
            # Blue gradient from light to dark
            intensity = 0.3 + 0.7 * (cluster_idx / max(1, total_clusters_in_branch - 1))
            return plt.cm.Blues(intensity)
        elif branch == "AD_branch":
            # Red gradient from light to dark
            intensity = 0.3 + 0.7 * (cluster_idx / max(1, total_clusters_in_branch - 1))
            return plt.cm.Reds(intensity)
        else:  # common_branch or any other
            return 'grey'

    # Create color mapping for each cluster based on its branch
    cluster_colors = {}
    for branch in df_tmp['branch'].unique():
        branch_clusters = cluster_centers[cluster_centers['branch'] == branch]['cluster'].sort_values().values
        total_in_branch = len(branch_clusters)
        
        for i, cluster in enumerate(branch_clusters):
            if cluster == 1:
                cluster_colors[cluster] = 'yellow'
            else:
                cluster_colors[cluster] = get_branch_color(branch, i, total_in_branch)

    # 1. Plot all points colored by their cluster's branch
    for cluster in clusters:
        cluster_data = df_tmp[df_tmp['cluster'] == cluster]
        ax.scatter(
            cluster_data["embedding1"], cluster_data["embedding2"],
            c=[cluster_colors[cluster]], s=4, alpha=0.6, linewidth=0
        )

    # 2. Plot cluster centers with same color as their points
    for _, row in cluster_centers.iterrows():
        cluster_color = cluster_colors[row["cluster"]]
        ax.scatter(row["embedding1"], row["embedding2"],
                color=cluster_color, edgecolor="k",
                s=35, marker="o", linewidth=0.5)
        
        # Use black text for cluster 4 (white background), white for others
        text_color = "white" if row["cluster"] == 4 else "white"
        ax.text(row["embedding1"], row["embedding2"], str(int(row["cluster"])),
                color=text_color, ha="center", va="center", fontsize=4.8, weight="bold")

    # 3. Connect cluster centers by branch
    branch_line_colors = {"common_branch": "grey", "AD_branch": "red", "normal_branch": "blue"}
    branch_labels = {"common_branch": None, "AD_branch": "AD branch", "normal_branch": "Normal branch"}
    for branch, branch_df in cluster_centers.groupby("branch"):
        branch_df = branch_df.sort_values("cluster")
        ax.plot(branch_df["embedding1"], branch_df["embedding2"],
                color=branch_line_colors.get(branch, "black"), linewidth=1.2, label=branch_labels[branch])

    # 4. Connect last common → first AD & first normal
    if "common_branch" in cluster_centers["branch"].values:
        last_common = cluster_centers[cluster_centers["branch"] == "common_branch"].sort_values("cluster").iloc[-1]
        if "AD_branch" in cluster_centers["branch"].values:
            first_ad = cluster_centers[cluster_centers["branch"] == "AD_branch"].sort_values("cluster").iloc[0]
            ax.plot([last_common["embedding1"], first_ad["embedding1"]],
                    [last_common["embedding2"], first_ad["embedding2"]],
                    color="red", linewidth=1.2)
        if "normal_branch" in cluster_centers["branch"].values:
            first_normal = cluster_centers[cluster_centers["branch"] == "normal_branch"].sort_values("cluster").iloc[0]
            ax.plot([last_common["embedding1"], first_normal["embedding1"]],
                    [last_common["embedding2"], first_normal["embedding2"]],
                    color="blue", linewidth=1.2)

    # 5. No axis ticks
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('Principal Component 1', fontsize=8)
    ax.set_ylabel('Principal Component 2', fontsize=8)

    # 6. Legend at top right
    handles = [plt.Line2D([0], [0], color=branch_line_colors[b], lw=2) for b in branch_line_colors]
    labels = list(branch_labels.values())
    ax.legend(handles[1:], labels[1:], loc="upper right", fontsize=6)

    fig.tight_layout()
    fig.show()

visualize_trajectory("train")

# --- Arrows ---

## Subjects

In [ ]:
import random
from matplotlib.lines import Line2D
from scipy import stats
import matplotlib.cm as cm

def draw_graph(dataset, seed=None): 
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test

    az = np.array([(x, y) for x, y in zip(df_tmp['embedding1'].values, df_tmp['embedding2'].values)])
    aRID = df_tmp['RID'].values
    alb = df_tmp['lb'].values
    aage = df_tmp['age'].values

    # Create plots
    fig, ax = plt.subplots(figsize=(FIGSIZE[0]*1.25, FIGSIZE[1]), dpi=DPI)

    # Draw background - training points
    scatter_train = ax.scatter(az[:, 0], az[:, 1], s=10, c="grey", linewidths=0, alpha=0.3)

    # Randomly select 2 unique RIDs from each label category in test data
    if seed is not None:
        random_seed = seed
    else:
        random_seed = random.randint(0, 10000)  # trial1-test: 7297
    print(f"Using random seed: {random_seed}")
    random.seed(random_seed)
    
    label_categories = {
        'CN': 1,
        'sMCI': [2, 3],
        'pMCI': [2, 3],
        'NCany': 1,
        'AD': 4,  # Moved AD to the end to ensure it's drawn last
    }
    
    selected_rids = []
    for label, value in label_categories.items():
        if isinstance(value, list):
            mask = np.isin(alb, value)
        else:
            mask = alb == value
        unique_rids = list(set(aRID[mask]))
        selected_rids.extend(random.sample(unique_rids, min(50, len(unique_rids))))

    mask = np.array([rid in selected_rids for rid in aRID])
    selected_z_test = az[mask]
    selected_age_test = aage[mask]
    selected_rid_test = np.array(aRID)[mask]
    selected_lb_test = np.array(alb)[mask]

    # Group points by RID and sort by age
    # Define color mapping for label pairs
    color_map = {
        'CN': 'c',       # Both first and last points have label 1
        'AD': 'm',         # Both first and last points have label 4
        'sMCI': 'orange',   # Both first and last points have label 2 or 3
        'pMCI': 'saddlebrown',      # First point has label 2 or 3, last point has label 4
        'NCany': 'g'        # Any other combination
    }
    
    # Draw arrows in specific order to ensure AD shows on top
    # First collect all trajectories grouped by category
    trajectories = {cat: [] for cat in color_map.keys()}
    
    for rid in selected_rids:
        # Get all points for this RID
        rid_mask = np.array([r == rid for r in selected_rid_test])
        rid_points = selected_z_test[rid_mask]
        rid_ages = selected_age_test[rid_mask]
        rid_labels = selected_lb_test[rid_mask]
        
        # Sort points by age
        sort_idx = np.argsort(rid_ages)
        rid_points_sorted = rid_points[sort_idx]
        rid_ages_sorted = rid_ages[sort_idx]
        rid_labels_sorted = rid_labels[sort_idx]
        
        # Get the label (first and last point)
        if len(rid_points_sorted) > 1:
            start_point = rid_points_sorted[0]
            end_point = rid_points_sorted[-1]
            first_label = rid_labels_sorted[0]
            last_label = rid_labels_sorted[-1]
            
            # Determine category based on first and last label
            if first_label == 1 and last_label == 1:
                category = 'CN'
            elif first_label == 4 and last_label == 4:
                category = 'AD'
            elif (first_label in [2, 3]) and (last_label in [2, 3]):
                category = 'sMCI'
            elif (first_label in [2, 3]) and last_label == 4:
                category = 'pMCI'
            elif first_label == 1 and (last_label in [2, 3, 4]):
                category = 'NCany'
            else:
                # print(f"Unknown label combination for RID {rid}: {first_label}, {last_label}")
                continue
                
            # Store trajectory data
            trajectories[category].append((rid_points_sorted, rid_ages_sorted, rid_labels_sorted))
    
    # Draw trajectories in specific order to ensure AD is on top
    draw_order = ['sMCI', 'pMCI', 'CN', 'AD']  # AD will be drawn last
    
    for category in draw_order:
        color = color_map[category]
        for rid_points_sorted, rid_ages_sorted, rid_labels_sorted in trajectories[category]:
            # Draw arrows connecting points in sequence
            for i in range(len(rid_points_sorted) - 1):
                ax.arrow(rid_points_sorted[i, 0], rid_points_sorted[i, 1],
                         rid_points_sorted[i+1, 0] - rid_points_sorted[i, 0],
                         rid_points_sorted[i+1, 1] - rid_points_sorted[i, 1],
                         width=0.08, head_starts_at_zero=True, fc=color, ec=color, alpha=0.7)
    
    # Create legend elements
    legend_elements = [
        Line2D([0], [0], color=color_map['CN'], lw=2, label='CN'),
        Line2D([0], [0], color=color_map['AD'], lw=2, label='AD'),
        Line2D([0], [0], color=color_map['sMCI'], lw=2, label='sMCI'),
        Line2D([0], [0], color=color_map['pMCI'], lw=2, label='pMCI'),
        # Line2D([0], [0], color=color_map['NCany'], lw=2, label='NCany')
    ]

    # Add legend for start and end points
    ax.legend(handles=legend_elements, loc='upper right', fontsize='small', handlelength=1.5, bbox_to_anchor=(1, 1))

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Arrows Plot ({dataset})')
        
    fig.tight_layout()
    fig.show()
    
draw_graph('train', 3467)  # 3467
draw_graph('test', None)  # 

## Highlighted subject

In [ ]:
def draw_subjects_arrows(dataset, RIDs=[]):
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(FIGSIZE[0]*1.25, FIGSIZE[1]), dpi=DPI)
    # Plot all points as grey background
    ax.scatter(df_tmp['embedding1'], df_tmp['embedding2'],
            c='lightgrey', alpha=0.3, s=8, linewidths=0, label='All data')
    
    for rid in RIDs:
        # Get all data points for this subject
        subject_data = df_tmp[df_tmp['RID'] == rid].copy()

        # Sort by age to connect points in chronological order
        subject_data = subject_data.sort_values('age')
        
        # Plot the trajectory points colored by label
        for i in subject_data['lb'].unique():
            mask = subject_data['lb'] == i
            if mask.any():
                ax.scatter(subject_data[mask]['embedding1'], subject_data[mask]['embedding2'],
                        c=visit_colors[i], alpha=0.8, s=20, linewidths=0, 
                        edgecolors='black', label=f'{labels[i]}')
        
        # Connect points in age order
        # ax.plot(subject_data['embedding1'], subject_data['embedding2'], 
        #         'k-', alpha=0.6, linewidth=1.5, label='Trajectory')
        
        # Add arrows to show direction of progression
        for i in range(len(subject_data) - 1):
            x1, y1 = subject_data.iloc[i]['embedding1'], subject_data.iloc[i]['embedding2']
            x2, y2 = subject_data.iloc[i+1]['embedding1'], subject_data.iloc[i+1]['embedding2']
            ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                        arrowprops=dict(arrowstyle='->', color='red', alpha=1, lw=1, label='Progression'))

        # Print subject details
        # print(f"Subject {rid} details:")
        # print(f"Age range: {subject_data['age'].min():.1f} - {subject_data['age'].max():.1f}")
        # print(f"Labels: {sorted(subject_data['lb'].unique())}")
        # print(f"Branches: {sorted(subject_data['branch'].unique())}")
        # print(f"Number of visits: {len(subject_data)}")

    ax.set_title(f'Subject Trajectory ({dataset})', fontsize=TITLE_SIZE)

    # Remove scales (tick marks and labels) from both axes
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(fontsize=8, loc='upper right')
    
    fig.tight_layout()
    fig.show()

In [ ]:
# Find subjects (Train)

subject_branches = df.groupby('RID')['branch'].unique()
subject_visits = df.groupby('RID').size()
subject_labels = df.groupby('RID')['lb'].unique()
subjects_with_both_branches = []

for rid, branches in subject_branches.items():
    if ('AD_branch' in branches 
        and subject_visits[rid] >= 3 
        and subject_labels[rid].size > 1
        and -1 not in subject_labels[rid]):  # Ensure multiple labels and no -1
        subjects_with_both_branches.append(rid)

print(f"Found {len(subjects_with_both_branches)} subjects with both branch types")

if len(subjects_with_both_branches) > 0:
    # Choose the first subject
    chosen_subject = subjects_with_both_branches[27]
    draw_subjects_arrows('train', RIDs=[chosen_subject])
    print(f"Chosen subject RID: {chosen_subject}")
else:
    print("No subjects founded")
    
"""
Training:

CN: 74, 413
AD: 2380, 4015
EMCI: 
LMCI: 
"""

In [ ]:
draw_subjects_arrows("train", RIDs=[413, 2380])

In [ ]:
# Find subjects (Test)

subject_branches = df_test.groupby('RID')['branch'].unique()
subject_visits = df_test.groupby('RID').size()
subject_labels = df_test.groupby('RID')['lb'].unique()
subjects_with_both_branches = []

for rid, branches in subject_branches.items():
    if ('normal_branch' in branches 
        and subject_visits[rid] >= 3
        and subject_labels[rid].size == 1
        and -1 not in subject_labels[rid]):  # Ensure multiple labels and no -1
        subjects_with_both_branches.append(rid)

print(f"Found {len(subjects_with_both_branches)} subjects with both branch types")

if len(subjects_with_both_branches) > 0:
    # Choose the first subject
    chosen_subject = subjects_with_both_branches[21]
    print(f"Chosen subject: {chosen_subject}")
    draw_subjects_arrows('test', RIDs=[chosen_subject])
else:
    print("No subjects founded")
    
"""
Test:

CN: 4043, 4469
AD: 4507, 4862
EMCI: 
LMCI: 
"""

In [ ]:
draw_subjects_arrows("test", RIDs=[4043, 4862])

# --- Stability ---

In [ ]:
def find_unstable_subjs(dataset):
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test

    # Find subjects with unstable trajectories
    # If subject have both AD_branch and normal_branch, consider it unstable
    unstable_subjects = []
    for rid in df_tmp['RID'].unique():
        subject_data = df_tmp[df_tmp['RID'] == rid]
        if 'AD_branch' in subject_data['branch'].values and 'normal_branch' in subject_data['branch'].values and -1 not in subject_data['lb'].values:
            unstable_subjects.append(rid)

    return unstable_subjects

unstable_RIDs_train = find_unstable_subjs('train')
print(f"Number of unstable subjects in train: {len(unstable_RIDs_train)}")
print(f"Unstable subjects in train: {unstable_RIDs_train}")
draw_subjects_arrows('train', unstable_RIDs_train)

# unstable_RIDs_test = find_unstable_subjs('test')
# print(f"Unstable subjects in test: {unstable_RIDs_test}")
# draw_subjects_arrows('test', [unstable_RIDs_test[2]])
# unstable_RIDs_test[2]

In [ ]:
len(unstable_RIDs_train)

# --- Pseudotime ---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.cm import ScalarMappable


def fade_unused_cmap(cmap_name, used_min, used_max, vmin=0, vmax=1, n=256, unused_alpha=0.15):
    base_cmap = plt.get_cmap(cmap_name)
    colors = base_cmap(np.linspace(0, 1, n))

    values = np.linspace(vmin, vmax, n)
    unused_mask = (values < used_min) | (values > used_max)

    colors[unused_mask, 3] = unused_alpha

    return ListedColormap(colors)


def plot_by_pseudotime(dataset, branch):
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test
    else:
        raise ValueError("dataset should be 'train' or 'test'")

    fig, ax = plt.subplots(figsize=(FIGSIZE[0] * 1.25, FIGSIZE[1]), dpi=DPI)

    ax.scatter(
        df_tmp['embedding1'], df_tmp['embedding2'],
        c='lightgrey', alpha=0.3, s=8, linewidths=0, label='All data'
    )

    ptime_col = f'ptime_{branch}'
    ptime_mask = df_tmp[ptime_col].notnull() & (df_tmp[ptime_col] != -1)

    if branch == 'AD':
        cmap_name = 'Reds'
    elif branch == 'normal':
        cmap_name = 'Blues'
    else:
        raise ValueError("branch should be 'AD' or 'normal'")

    ptime_values = df_tmp.loc[ptime_mask, ptime_col]

    used_min = ptime_values.min()
    used_max = ptime_values.max()

    norm = Normalize(vmin=0, vmax=1)

    # points keep original colors
    ax.scatter(
        df_tmp.loc[ptime_mask, 'embedding1'],
        df_tmp.loc[ptime_mask, 'embedding2'],
        c=ptime_values,
        cmap=cmap_name,
        norm=norm,
        alpha=0.7,
        s=10,
        linewidths=0
    )

    # colorbar uses faded unused range
    fade_cmap = fade_unused_cmap(
        cmap_name,
        used_min=used_min,
        used_max=used_max,
        vmin=0,
        vmax=1,
        unused_alpha=0.0
    )

    sm = ScalarMappable(norm=norm, cmap=fade_cmap)
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=ax)
    # cbar.set_label('Pseudotime', fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    ax.set_title(f'Pseudotime (common + {branch} branch)', fontsize=TITLE_SIZE)
    ax.set_xlabel('PC1', fontsize=8)
    ax.set_ylabel('PC2', fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

    fig.tight_layout()
    plt.show()

plot_by_pseudotime('test', 'normal')
plot_by_pseudotime('test', 'AD')

# --- Classification basis ---

In [ ]:
# Draw two figures for CN and AD class

def plot_by_class(dataset, class_label):
    if dataset == 'train':
        df_tmp = df
    elif dataset == 'test':
        df_tmp = df_test

    fig, ax = plt.subplots(figsize=(FIGSIZE[0]*1.25, FIGSIZE[1]), dpi=DPI)

    # Plot all points as grey background
    ax.scatter(df_tmp['embedding1'], df_tmp['embedding2'],
               c='lightgrey', alpha=0.3, s=8, linewidths=0, label='All data')

    if class_label == 'AD':
        df_colored = df_tmp[df_tmp['branch'] == 'AD_branch']
        color = 'red'
    elif class_label == 'CN':
        df_colored = df_tmp[(df_tmp['branch'] == 'normal_branch') | (df_tmp['branch'] == 'common_branch')]
        color = 'blue'

    scatter = ax.scatter(df_colored['embedding1'], df_colored['embedding2'],
               c=color, alpha=0.7, s=10, linewidths=0, vmin=0, vmax=1)

    # Remove scales (tick marks and labels) from both axes
    ax.set_xlabel(f'Class: {class_label}')
    ax.set_xticks([])
    ax.set_yticks([])

    fig.tight_layout()
    fig.show()

plot_by_class('train', 'CN')
plot_by_class('train', 'AD')